In [1]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: C:\Users\amanp\OneDrive\Desktop\transformers\venv\Scripts\python.exe -m pip install --upgrade pip


In [1]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis')
result = classifier('I was so happy with the mission impossible movie')
print(result)

C:\Users\amanp\OneDrive\Desktop\transformers\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3117.88it/s]


[{'label': 'POSITIVE', 'score': 0.9998778104782104}]


In [4]:
# tokenization


# "I love India."
#         ↓
# Tokenization by tokenizer
#         ↓
# ["I", "love", "India", "."]
#         ↓
# Token IDs
#         ↓
# [1045, 2293, 2634, 1012]
#         ↓
# Tensor
#         ↓
# Model

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased') # downloads and loads the tokenizer that was used to train bert-base-uncased

In [7]:
text = ' I love India'
output = tokenizer(text)

output

{'input_ids': [101, 1045, 2293, 2634, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [10]:
# input_ids => token id

# 101 → [CLS]
# 1045 → I
# 2293 → love
# 2634 → India
# 102 → [SEP]


# Sentence A:
# I love India.

# Sentence B:
# It is a beautiful country.

# [CLS] I love India [SEP] It is a beautiful country [SEP]

## token_type_ids : which sent does this token belong to?

# input_ids:
# [CLS] I love India [SEP] It is beautiful [SEP]

# token_type_ids:
# 0     0  0    0     0    1  1   1        1


# 'attention_mask': (bert/gpt)
# [1, 1, 1, 1, 1]  
# tells which tokens are realm and which are jsut padding?
# 1 = can attend -> real token
# 0 = blocked -> padding token

In [11]:
# casual mask (gpt)
# future toekn -> Prevent the model from looking into the future.


In [ ]:
# encode() 
# hf return dict because models need multiple things: input_ids, attention_mask, token_type_ids
# but sometimes only want the token ids, then we use encode()




In [12]:
ids = tokenizer.encode('I Love India.')
ids

[101, 1045, 2293, 2634, 1012, 102]

In [17]:
text = tokenizer.decode(ids)
text     # output lower case

'[CLS] i love india. [SEP]'

In [18]:
text = tokenizer.decode(ids, skip_special_tokens = True)
text

'i love india.'

In [20]:
# feature extraction
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased') # loads the tokenizer
model = AutoModel.from_pretrained('bert-base-uncased') # loads the pretrained nn

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 628.95it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
text = 'I love India.'
inputs = tokenizer(text, return_tensors='pt') # return the output as pytorch tensors
print(inputs)

{'input_ids': tensor([[ 101, 1045, 2293, 2634, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [25]:
# tensor([[ 101, 1045, 2293, 2634, 1012,  102]])   -> [[]] double brackets instead [].
# shape : (1,5)  => 1: batch size, 5 : seq length

# cause pytorch models expect batches, not individual samples
# even for a songle sent , hf -> batch size =1

#cause during training we don't process 1 sent -> we process many sent together

In [31]:
# inputs = {
#     "input_ids": tensor([[101, 1045, 2293, 2634, 102]]),
#     "attention_mask": tensor([[1, 1, 1, 1, 1]])
# }

In [33]:
outputs = model(**inputs)
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-1.7282e-01,  3.9364e-01, -1.6543e-01,  ..., -3.8659e-01,
           4.9403e-01,  4.9169e-01],
         [ 4.6276e-01,  4.8760e-01, -6.2829e-01,  ...,  1.3227e-01,
           1.0235e+00,  5.0731e-01],
         [ 1.2162e+00,  9.4970e-01,  5.4405e-01,  ..., -5.4646e-01,
           8.3622e-01,  3.2712e-01],
         [-2.6962e-01, -3.2546e-01, -6.8879e-01,  ..., -3.1257e-01,
          -5.9101e-04,  5.6867e-02],
         [ 3.3074e-01, -2.7828e-01, -8.4807e-02,  ...,  5.7126e-01,
           7.1686e-01, -2.7556e-01],
         [ 7.7489e-01,  4.3880e-02,  4.6193e-02,  ..., -1.4323e-01,
          -3.8772e-01, -3.7270e-01]]], grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-0.9184, -0.4651, -0.9037,  0.8272,  0.7839, -0.2137,  0.8572,  0.3012,
         -0.8301, -1.0000, -0.5021,  0.9754,  0.9848,  0.5734,  0.9250, -0.7889,
         -0.5182, -0.6056,  0.4267, -0.5678,  0.7553,  1.0000, -0.0752,  0.2777,
          0

In [34]:
# ** means -> unpack the dict into keyword arguments

# Input IDs
#       ↓
# Embeddings
#       ↓
# Transformer Layer 1
#       ↓
# Transformer Layer 2
#       ↓
# ...
#       ↓
# Transformer Layer 12
#       ↓
# last_hidden_state   -> output of the last transformer layer

In [36]:
print(outputs.last_hidden_state.shape)   # (batch_size, input_ids, embed_dim)

torch.Size([1, 6, 768])


In [ ]:
# Raw Text
#     ↓
# AutoTokenizer
#     ↓
# input_ids
# attention_mask
# token_type_ids
#     ↓
# return_tensors="pt"
#     ↓
# AutoModel
#     ↓
# last_hidden_state
# (batch_size, seq_len, hidden_size)

In [41]:
# pooler_output
# It is a single 768-dimensional vector representing the whole sentence.

print(outputs.pooler_output.shape)

torch.Size([1, 768])


In [42]:
# last_hidden_state -> (1,6,768)


# Batch 1

# [CLS]  → (768,)
# I      → (768,)
# love   → (768,)
# India  → (768,)
# .      → (768,)
# [SEP]  → (768,)   

# 6 vectors


# but for tasks like sentiment analysis, spam detection, classification
# we need one vect representing the entire sent
# so bert takes the [cls] token's representaion ( after passing it through a small dense layer + tahh) and returns:
#pooler_output -> (1,768)

In [47]:
# padding
# needed cause -> Because tensors require every row to have the same length.

texts = [
    "I love India.",
    "Hello",
    "Transformers are amazing."
]

output = tokenizer(
    texts,
    padding=True,  # [PAD] → 0 <- token ids
    return_tensors="pt"
)

print(output)

{'input_ids': tensor([[  101,  1045,  2293,  2634,  1012,   102],
        [  101,  7592,   102,     0,     0,     0],
        [  101, 19081,  2024,  6429,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1]])}


In [49]:
# bert has max seq len = 512
# so it can't process > 512 tokens.

# solution -> truncate
# 800 tokens
#       ↓
# Keep first 512
#       ↓
# Discard remaining 288

output = tokenizer(
    text,
    truncation = True,
    max_length = 512,
    return_tensors = 'pt'
)
output

{'input_ids': tensor([[ 101, 1045, 2293, 2634, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [50]:
# can we use the rest seq in row2, instead discard?

# case 1: classification(sentiment analysis)
# input : the movie was amazing... (800 tokens)
# output : positive

# split into :
# chunk1 -> positive?
# chunk2 -> positive?
# which one will be final ans???
# model was supposed to classify the entire document, not two independent chunks




# case2: question answering
# for long documents, we can split

# Document
#       ↓
# Chunk 1
# Chunk 2
# Chunk 3
#       ↓
# Run model on each
#       ↓
# Combine results

# here the idea is correct



# case3: modern llms 
# they don't dicard cause they support much larger context windows

In [52]:
# why hf use truncation?

# cause the model has a max input size
# bert was trained with -> max = 512 tokens
# if we give 800 tokens -> its positonal embedding only go up to 512
# model literally can't process toekn 513

In [57]:
# feature extraction

outputs = model(**inputs)
features = outputs.last_hidden_state

# those features are exactly -> feature extraction means

# input : 'I love India'
# model con't use raw words as features

# Text
#    ↓
# Tokenizer
#    ↓
# BERT
#    ↓
# last_hidden_state

# each token -> 512 dim vector

# "I"
# ↓
# [-0.17, 0.39, ..., 0.49]   (768 numbers)
# this vec captues the mean and context of the token
# that vector is called feature rep or features


# why called it a feature?
# cause they describe the token in its context, not physically but semantically